# Evaluating GenAI output — LLM-as-judge

*Part of the `gen_ai/` track. Prerequisite: `a_tracing_quickstart`. The traditional-ML analog is `ml/e_model_evaluation`.*

## The problem: there's no RMSE for "a good answer"

In `ml/e_model_evaluation` you scored a regressor with RMSE / R² — a **formula** over numbers. A GenAI answer has no such formula: *is this response relevant, correct, on-policy, free of nonsense?* is a **judgment**, not arithmetic.

MLflow's answer is the **LLM-as-judge**: a capable LLM reads each *(input, output)* pair and scores it against a criterion. `mlflow.genai.evaluate()` runs one or more **scorers** over a dataset and logs the scores as metrics plus per-row **assessments** — the GenAI analog of `mlflow.models.evaluate()`.

| Term | One-line meaning |
|---|---|
| **scorer** | A function that grades one row. Built-in (`RelevanceToQuery`, `Correctness`, …) or your own `@scorer`. |
| **judge** | The LLM a scorer calls to make the grade. Must be *capable* — a weak model judges unreliably. |
| **assessment** | The per-row result (score + the judge's rationale), attached to the row's trace in the UI. |

## Prerequisites

**1. A tracking server on port 5001** (from `src/`, separate terminal): `mlflow server --host 127.0.0.1 --port 5001`.

**2. A capable judge model.** LLM-as-judge only works if the judge is strong enough to grade reliably — a *small* local model is too weak and gives noisy scores. Pick one:

- **Azure OpenAI** (used here): URI `azure:/<deployment>`. The built-in judges read `AZURE_API_KEY` / `AZURE_API_BASE` / `AZURE_API_VERSION`, which we map from the repo's `AZURE_OPENAI_*` `.env` values in the next cell.
- **OpenAI**: `openai:/gpt-4.1-mini` (MLflow's default judge) — set `OPENAI_API_KEY`.
- **A large local model**: `uv add litellm`, then a litellm URI like `ollama/qwen3:14b`. Small local models are not recommended *as judges*.

The app being graded can still be anything (we generate some answers with local Ollama at the end) — only the **judge** needs to be capable.

**Diverges from upstream tutorial:** MLflow's eval docs assume the Databricks managed judge or OpenAI. We route the judge to **Azure** (a capable hosted model the repo already has access to) through MLflow's native `azure` provider — no `litellm` needed.

## Point the judge at Azure

MLflow's `azure` judge provider expects `AZURE_API_KEY` / `AZURE_API_BASE` / `AZURE_API_VERSION`. Our `.env` uses the standard `AZURE_OPENAI_*` names, so we map them once. `AZURE_API_BASE` is the **resource root** (without the trailing `/openai`).

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # read the gitignored .env at the repo root

# Map the repo's AZURE_OPENAI_* names to what MLflow's azure judge provider reads.
os.environ["AZURE_API_KEY"]     = os.environ["AZURE_OPENAI_API_KEY"]
os.environ["AZURE_API_BASE"]    = os.environ["AZURE_OPENAI_BASE_URL"].removesuffix("/openai").rstrip("/")
os.environ["AZURE_API_VERSION"] = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21")

import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("genai-evaluation")

# The judge: any capable deployment. nano is plenty for these examples; use a bigger
# deployment for harder, more subjective judgments.
JUDGE = f"azure:/{os.environ.get('AZURE_OPENAI_LIGHT_MODEL', 'gpt-5.4-nano')}"
print("judge:", JUDGE)

## A tiny evaluation set

Each row is a dict with **`inputs`** (what the app was asked) and **`outputs`** (what it answered). We include obviously good and obviously bad answers so you can see the judge discriminate.

In [ ]:
data = [
    {"inputs": {"question": "What is the capital of France?"},
     "outputs": "The capital of France is Paris."},
    {"inputs": {"question": "What is the capital of France?"},
     "outputs": "I really enjoy hiking in the mountains on weekends."},
    {"inputs": {"question": "Who wrote Hamlet?"},
     "outputs": "Hamlet was written by William Shakespeare."},
]

## Score it with built-in judges

`RelevanceToQuery` asks *does the answer address the question?* `Guidelines` checks the answer against a free-text rule you write. Each takes `model=JUDGE`.

In [ ]:
from mlflow.genai.scorers import RelevanceToQuery, Guidelines

results = mlflow.genai.evaluate(
    data=data,
    scorers=[
        RelevanceToQuery(model=JUDGE),
        Guidelines(name="concise",
                   guidelines="The response is a single sentence with no filler.",
                   model=JUDGE),
    ],
)
results.metrics

`relevance_to_query/mean` should land around **0.67** — two of the three answers are on-topic, the hiking one isn't. Open <http://127.0.0.1:5001> → the experiment → **Evaluation runs** to see the per-row scores, and click a row to read the **judge's rationale** in its trace assessment. That rationale is why LLM-as-judge beats a bare number: it tells you *why* a row failed.

## Reference-based scoring with `Correctness`

When you have a known-good answer, add it as an **`expectations`** field and use `Correctness` — the judge checks whether the output's facts match the expected ones.

In [ ]:
from mlflow.genai.scorers import Correctness

graded = [
    {"inputs": {"question": "What is 2 + 2?"}, "outputs": "It is 4.",
     "expectations": {"expected_response": "4"}},
    {"inputs": {"question": "What is 2 + 2?"}, "outputs": "It is 5.",
     "expectations": {"expected_response": "4"}},
]
mlflow.genai.evaluate(data=graded, scorers=[Correctness(model=JUDGE)]).metrics

## Your own criteria with `@scorer`

Built-ins don't cover every rule. A `@scorer` function grades a row with *any* Python — deterministic (length, regex, a PII check) or its own LLM call. It receives whichever of `inputs` / `outputs` / `expectations` you name as parameters, and returns a bool, number, or `Feedback`.

In [ ]:
from mlflow.genai.scorers import scorer

@scorer
def under_120_chars(outputs) -> bool:
    # a cheap, deterministic guardrail — no judge LLM involved
    return len(str(outputs)) <= 120

# Custom and built-in scorers run side by side.
mlflow.genai.evaluate(
    data=data,
    scorers=[under_120_chars, RelevanceToQuery(model=JUDGE)],
).metrics

## Evaluating a real app (generate, then judge)

So far we scored *static* answers. To evaluate an actual app, pass a **`predict_fn`** and rows of `inputs` only — MLflow calls `predict_fn(**inputs)` to generate each output, then scores it. Here the app is the local Ollama call from `a_tracing_quickstart` (free to generate), judged by Azure (capable). Generating locally + judging with a hosted model is a common, cost-aware split.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

def app(question: str) -> str:
    r = client.chat.completions.create(
        model="qwen3:8b",
        messages=[{"role": "user", "content": f"{question} /no_think"}],
    )
    return r.choices[0].message.content.strip()

eval_set = [
    {"inputs": {"question": "In one sentence, what is MLflow Tracing?"}},
    {"inputs": {"question": "In one sentence, what is an MLflow model registry?"}},
]
mlflow.genai.evaluate(
    data=eval_set,
    predict_fn=app,
    scorers=[RelevanceToQuery(model=JUDGE)],
).metrics

## Next steps

- **`d_prompt_registry.ipynb`** — now that you can *score* an app, you can compare two **prompt versions** and keep the winner. `register_prompt`, versions + aliases — the GenAI analog of `ml/f_model_registry`.
- Wire a scorer into CI as a **gate** (fail the build if `relevance_to_query/mean` drops), exactly as `ml/e_model_evaluation` gates a regressor with `validate_evaluation_results`.